# Plot Constructor Example

This notebook demonstrates a decomposed pipeline: user parameters -> data loading/preprocessing -> plotting.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from typing import Any
import numpy as np
import pandas as pd

from app.visualization import PlotConstructor

## 1) User parameters

Set date, ionosonde station codes, and which plots should be built.

In [ ]:
# --- User-configurable parameters ---
USER_PARAMS = {
    "date_str": "2025-11-12",
    "ionosonde_codes": ["mau", "arl"],
    "plot_requests": [
        "ROTI",
        "Dst",
        "Kp",
    ],
}

USER_PARAMS

## 2) Data preparation pipeline

A small decomposed pipeline that downloads/loads only required data and returns `processor_results` for `PlotConstructor`.

In [ ]:
from datetime import datetime, timedelta
from app.gfz.gfz_downloader import GfzDownloader
from app.gfz.gfz_processor import GfzProcessor
from app.ionosonde.ionosonde_downloader import IonosondeDownloader
from app.ionosonde.ionosonde_processor import IonosondeProcessor
from app.kyoto.kyoto_dst_downloader import KyotoDstDownloader
from app.kyoto.kyoto_dst_processor import KyotoProcessor
from app.nmdb.nmdb_downloader import NmdbDownloader
from app.nmdb.nmdb_processor import NmdbProcessor
from app.omni.omni_downloader import OmniDownloader
from app.omni.omni_processor import OmniProcessor
from app.pipeline.observation_pipeline import (
    collect_observation_links,
    load_observations_from_csv,
    parse_and_save_observations,
)
from app.simurg.gim_downloader import GimDownloader
from app.simurg.gim_processor import GimProcessor
from app.simurg.simurg_client import SimurgClient
from app.simurg.simurg_downloader import AdjustedTecDownloader, RotiDownloader
from app.simurg.simurg_processor import DataProduct, SimurgProcessor
from app.storage.hdf5_storage import ObservationHDF5Storage


def _safe_step(step_name: str, fn):
    try:
        return fn()
    except Exception as exc:
        print(f"[WARN] {step_name} failed: {exc}")
        return None


def load_all_results(params: dict[str, Any]) -> dict[str, Any]:
    """Сначала скачиваем, затем обрабатываем все доступные наборы данных."""
    date_str = params["date_str"]
    ionosonde_station = (params.get("ionosonde_codes") or [None])[0]

    base_dir = Path.cwd().parent
    download_dir = base_dir / "files" / date_str
    download_dir.mkdir(parents=True, exist_ok=True)

    simurg_dir = download_dir / "simurg"
    gim_dir = download_dir / "gim"
    omni_dir = download_dir / "omni"
    kp_dir = download_dir / "kp"
    dst_dir = download_dir / "kyoto"
    nmdb_dir = download_dir / "nmdb"
    ionosonde_dir = download_dir / "ionosonde"

    for p in [simurg_dir, gim_dir, omni_dir, kp_dir, dst_dir, nmdb_dir, ionosonde_dir]:
        p.mkdir(parents=True, exist_ok=True)

    target_date = datetime.strptime(date_str, "%Y-%m-%d")
    start_date = target_date - timedelta(days=15)
    end_date = target_date + timedelta(days=15)

    # 1) DOWNLOAD
    simurg_client = SimurgClient(email="Storm_Plotter_Jupyter_Notebook@gmail.com")
    _safe_step("Adjusted TEC download", lambda: AdjustedTecDownloader(client=simurg_client, out_dir=str(simurg_dir)).download(date_str))
    _safe_step("ROTI download", lambda: RotiDownloader(client=simurg_client, out_dir=str(simurg_dir)).download(date_str))
    _safe_step("GIM download", lambda: GimDownloader(out_dir=str(gim_dir)).download(date_str))

    _safe_step("OMNI download", lambda: OmniDownloader(out_dir=str(omni_dir)).download(date_str))
    _safe_step("Kp download", lambda: GfzDownloader(out_dir=str(kp_dir)).download(start_date=start_date, end_date=end_date))
    _safe_step("Dst download", lambda: KyotoDstDownloader(out_dir=str(dst_dir)).download(date_str))

    _safe_step("NMDB download", lambda: NmdbDownloader(out_dir=str(nmdb_dir)).download(start=start_date, end=end_date, stations=None))
    _safe_step("Ionosonde download", lambda: IonosondeDownloader(out_dir=str(ionosonde_dir)).download(target_date=date_str, station=ionosonde_station))

    h5_path = str(download_dir / "spaceweather_observations.h5")
    csv_path = str(download_dir / "aurora_data.csv")
    date_slash = date_str.replace("-", "/")
    _safe_step("Aurora links collection", lambda: collect_observation_links(date_slash, h5_path))

    # 2) PROCESS
    results: dict[str, Any] = {}

    results["Adjusted TEC"] = _safe_step(
        "Adjusted TEC processing",
        lambda: SimurgProcessor(str(simurg_dir)).load(date_str, product_type=DataProduct.TEC_ADJUSTED),
    )
    results["ROTI"] = _safe_step(
        "ROTI processing",
        lambda: SimurgProcessor(str(simurg_dir)).load(date_str, product_type=DataProduct.ROTI),
    )
    results["Keogram"] = results["ROTI"]
    results["GIM"] = _safe_step("GIM processing", lambda: GimProcessor(str(gim_dir)).load(date_str))

    omni_df = _safe_step("OMNI processing", lambda: OmniProcessor(str(omni_dir)).load(date_str))
    if omni_df is not None:
        omni_expected = set(OmniDownloader.OMNI_1MIN_VARS.keys())
        present = sorted(omni_expected.intersection(set(omni_df.columns.str.lower())))
        print(f"OMNI columns available ({len(present)}/{len(omni_expected)}): {present}")
    results["OMNI"] = omni_df

    results["Dst"] = _safe_step("Dst processing", lambda: KyotoProcessor(str(dst_dir)).load(date_str))
    results["Kp"] = _safe_step("Kp processing", lambda: GfzProcessor(str(kp_dir)).load(date_str=date_str))

    results["Cosmic Ray"] = _safe_step("Cosmic Ray processing", lambda: NmdbProcessor(str(nmdb_dir)).load(date_str))
    results["Ionosonde"] = _safe_step(
        "Ionosonde processing",
        lambda: IonosondeProcessor(str(ionosonde_dir)).load(target_date=date_str, station=ionosonde_station),
    )

    observations = []
    observations.extend(load_observations_from_csv(csv_path, date_str))

    storage = ObservationHDF5Storage(h5_path)
    if storage.has_date(date_str):
        parsed = _safe_step(
            "Aurora parsing",
            lambda: parse_and_save_observations(h5_path, csv_path, dates=[date_str]),
        )
        observations.extend(parsed or [])

    results["Aurora"] = pd.DataFrame(observations) if observations else None

    return {k: v for k, v in results.items() if v is not None}


processor_results = load_all_results(USER_PARAMS)
processor_results.keys()


## 3) Build plots with PlotConstructor

In [ ]:
plotter = PlotConstructor(processor_results)
plotter.available_plots()

In [ ]:
fig, _ = plotter.plot(USER_PARAMS["plot_requests"])
fig

## 4) Optional: per-plot parameters

In [ ]:
if "ROTI" in processor_results and isinstance(processor_results["ROTI"], dict):
    first_roti_time = sorted(processor_results["ROTI"].keys())[0]
    fig, _ = plotter.plot([
        {"name": "ROTI", "params": {"plot_time": first_roti_time, "cmap": "viridis", "s": 10}},
        {"name": "Kp"},
        {"name": "Dst", "params": {"color": "black"}},
    ])
    fig